# Muse EEG studies of meditation and attention

EEG studies often compare band power between defined conditions, track changes during practice, and check whether the results survive artifact screening. For concentration and mind wandering, researchers can ask participants to report what they were attending to at specific times. For example, Brandmeyer and Delorme used probes during seated practice to compare reported meditation and mind wandering. [Brandmeyer and Delorme, 2018](https://pubmed.ncbi.nlm.nih.gov/27815577/)

These three examples use your retained archive of directed-thought practice. The archive records the band values and their timing. A time window gives an approximate session phase, while a report from the practitioner would provide a label for focus, contemplation, or mind wandering. AF7 and AF8 are lateral frontal sensors. Studies of frontal **midline** theta measure a different location.

**Run All** to scan the archive and show the tables and plots, including three examples focused on alpha. The notebook uses the saved band columns in their original recorded units. It reads the CSVs locally and produces its results in memory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ARCHIVE = Path.home() / "Documents" / "Yijing" / "Muse"
STARTUP_SECONDS = 120
WIND_DOWN_SECONDS = 120
WINDOW_SECONDS = 10
MIN_SECONDS_PER_WINDOW = 8
TIME_BLOCK_SECONDS = 5 * 60
TIME_BLOCK_COUNT = 4
MIN_TIME_BLOCK_WINDOWS = 24
MIN_MOTION_BLOCK_WINDOWS = 20

BANDS = ("Delta", "Theta", "Alpha", "Beta", "Gamma")
SENSORS = ("TP9", "AF7", "AF8", "TP10")
EEG_COLUMNS = [f"{band}_{sensor}" for band in BANDS for sensor in SENSORS]
SENSOR_PAIRS = {"Front": ("AF7", "AF8"), "Rear": ("TP9", "TP10")}
PAIR_COLUMNS = [f"{band}_{pair}" for band in BANDS for pair in SENSOR_PAIRS]
GAMMA_DIFFERENCES = [f"Gamma_{right}_minus_{left}" for left, right in SENSOR_PAIRS.values()]
CONTRAST_COLUMNS = EEG_COLUMNS + PAIR_COLUMNS + GAMMA_DIFFERENCES
SENSOR_COLORS = dict(zip(SENSORS, ["tab:blue", "tab:purple", "tab:green", "tab:orange"]))
GYRO_COLUMNS = ["Gyro_X", "Gyro_Y", "Gyro_Z"]
FORMAT_LABELS = {"Mellow": "Mellow format", "PPG": "PPG format", "Other": "Other format"}

## Prepare the recordings

Use the timestamp inside each file and omit rows marked headset off. Take medians within each second and then within each 10-second window. Each channel needs eight usable seconds per window. Pair means and right-minus-left differences use readings available on both sides. A missing window stays missing.

The first two minutes are excluded because the direct Bluetooth code can use that time to establish its baseline. The headset stays on until recording stops. The full final two minutes are used as the eyes-open wind-down, with its start estimated from the file end. Mellow and PPG headers identify export formats. They are a proxy for recording settings, so read each format separately. The retained sessions come from one archive and include gaps and selection from earlier session deletions.

In [ ]:
def median_windows(per_second, anchor):
    starts = anchor + (per_second.index - anchor) // WINDOW_SECONDS * WINDOW_SECONDS
    groups = per_second.groupby(starts)
    return groups.median().where(groups.count() >= MIN_SECONDS_PER_WINDOW)


def read_windows(path):
    header = pd.read_csv(path, nrows=0).columns
    required = ["TimeStamp", *EEG_COLUMNS]
    missing = sorted(set(required) - set(header))
    if missing:
        raise ValueError(f"Missing columns: {', '.join(missing)}")

    recording_format = "PPG" if "PPG_IR" in header else "Mellow" if "Mellow" in header else "Other"
    numeric = [*EEG_COLUMNS, "HeadBandOn", *GYRO_COLUMNS]
    data = pd.read_csv(path, usecols=[c for c in ["TimeStamp", *numeric] if c in header],
                       low_memory=False)
    times = pd.to_datetime(data["TimeStamp"], format="mixed", errors="coerce")
    values = data.reindex(columns=numeric).apply(pd.to_numeric, errors="coerce")
    values = values.replace([np.inf, -np.inf], np.nan)
    timed_eeg = times.notna() & values[EEG_COLUMNS].notna().any(axis=1)
    if not timed_eeg.any():
        raise ValueError("No timestamped EEG readings.")

    start = times[timed_eeg].min()
    elapsed = (times - start).dt.total_seconds()
    end = int(elapsed[timed_eeg].max()) + 1  # Include the final second with EEG.
    usable = timed_eeg & ~values["HeadBandOn"].eq(0)
    kept = values.loc[usable]
    signals = kept[EEG_COLUMNS].copy()
    for pair, (left, right) in SENSOR_PAIRS.items():
        for band in BANDS:
            signals[f"{band}_{pair}"] = (kept[f"{band}_{left}"] + kept[f"{band}_{right}"]) / 2
        signals[f"Gamma_{right}_minus_{left}"] = kept[f"Gamma_{right}"] - kept[f"Gamma_{left}"]
    gyro = kept[GYRO_COLUMNS]
    if gyro.nunique().gt(1).any():
        signals["Gyro"] = np.sqrt(gyro.pow(2).sum(axis=1, min_count=3))
    else:
        signals["Gyro"] = np.nan

    per_second = signals.groupby(elapsed.loc[usable].astype(int)).median()
    per_second = per_second.loc[per_second.index >= STARTUP_SECONDS]
    windows = median_windows(per_second, anchor=STARTUP_SECONDS)
    phase_start = end - 2 * WIND_DOWN_SECONDS
    phase_windows = median_windows(per_second.loc[per_second.index >= phase_start], anchor=end)
    if windows[EEG_COLUMNS].dropna(how="all").empty:
        raise ValueError("No usable EEG windows after startup.")
    return start, recording_format, end, windows, phase_windows


if not ARCHIVE.is_dir():
    raise FileNotFoundError(f"Set ARCHIVE to the folder containing the Muse CSVs: {ARCHIVE}")

files = sorted(ARCHIVE.glob("*.csv"))
recordings, skipped = {}, []
for index, path in enumerate(files, start=1):
    try:
        recordings[path.name] = read_windows(path)
    except (OSError, ValueError) as error:
        skipped.append({"File": path.name, "Reason": str(error)})
    if index % 75 == 0:
        print(f"Read {index} of {len(files)} files.")

if not recordings:
    raise ValueError(f"No usable recordings. First skipped files: {skipped[:3]}")

inventory = pd.DataFrame(
    [(file, start, fmt, end / 60) for file, (start, fmt, end, _, _) in recordings.items()],
    columns=["File", "Started", "Format", "Minutes"],
).sort_values("Started")
print(f"Loaded {len(recordings)} of {len(files)} files.")
display(inventory.groupby("Format").agg(
    Sessions=("File", "size"), First=("Started", "min"), Last=("Started", "max")
))
if skipped:
    display(pd.DataFrame(skipped))

## 1. Compare practice with the eyes-open wind-down

Condition contrasts are common in meditation EEG. Lagopoulos and colleagues compared nondirective meditation with quiet rest while participants kept their eyes closed in both conditions. Cahn and colleagues compared Vipassana with an instructed mind-wandering condition in counterbalanced order. [Lagopoulos et al., 2009](https://pubmed.ncbi.nlm.nih.gov/19922249/) · [Cahn et al., 2010](https://pubmed.ncbi.nlm.nih.gov/20013298/)

Compare **the two minutes immediately before wind-down** with **the full two-minute eyes-open wind-down**. The windows are aligned to the final recorded EEG second so the comparison includes the end of the file. Each band and sensor needs at least 75% coverage in both stretches. The heatmaps show the median paired difference across sessions. Compare the signs across sensors and use the table to see how often the differences point in the same direction at each sensor and for the front and rear pairs. Eye state, time in the session, and stopping the practice can all contribute to this contrast.

In [ ]:
def slice_windows(windows, start, stop):
    return windows.loc[(windows.index >= start) & (windows.index + WINDOW_SECONDS <= stop)]


def summarize_contrast(frame, columns):
    return pd.DataFrame({
        "Sessions": frame[columns].count(),
        "Median change": frame[columns].median(),
        "25th percentile": frame[columns].quantile(0.25),
        "75th percentile": frame[columns].quantile(0.75),
        "% positive": 100 * frame[columns].gt(0).sum() / frame[columns].count(),
    }).round(3)


contrasts = []
minimum = int(np.ceil(0.75 * WIND_DOWN_SECONDS / WINDOW_SECONDS))
for file, (_, fmt, end, _, phase_windows) in recordings.items():
    boundary = end - WIND_DOWN_SECONDS
    practice_start = boundary - WIND_DOWN_SECONDS
    if practice_start < STARTUP_SECONDS:
        continue
    practice = slice_windows(phase_windows, practice_start, boundary)
    wind_down = slice_windows(phase_windows, boundary, end)
    eligible = practice.count().ge(minimum) & wind_down.count().ge(minimum)
    difference = (practice.median() - wind_down.median()).where(eligible)
    contrasts.append({"File": file, "Format": fmt, **difference.to_dict()})

phase_contrasts = pd.DataFrame(contrasts, columns=["File", "Format", *CONTRAST_COLUMNS])
for fmt, group in phase_contrasts.groupby("Format", sort=False):
    if group[EEG_COLUMNS].count().sum() == 0:
        continue
    print(FORMAT_LABELS[fmt])
    display(summarize_contrast(group, CONTRAST_COLUMNS))
    matrix = group[EEG_COLUMNS].median().to_numpy().reshape(len(BANDS), len(SENSORS))
    counts = group[EEG_COLUMNS].count().to_numpy().reshape(matrix.shape)
    limit = max(np.nanmax(np.abs(matrix)), 0.01)
    fig, ax = plt.subplots(figsize=(6.8, 3.9), layout="constrained")
    heatmap = ax.imshow(matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.set(xticks=range(len(SENSORS)), xticklabels=SENSORS,
           yticks=range(len(BANDS)), yticklabels=BANDS,
           title=f"{FORMAT_LABELS[fmt]} | Late practice minus wind-down")
    for (band, sensor), value in np.ndenumerate(matrix):
        if np.isfinite(value):
            color = "white" if abs(value) > 0.65 * limit else "black"
            ax.text(sensor, band, f"{value:+.2f}\nn={counts[band, sensor]}",
                    ha="center", va="center", color=color, fontsize=9)
    fig.colorbar(heatmap, ax=ax, label="Recorded band value difference")
    plt.show()
if phase_contrasts[EEG_COLUMNS].notna().sum().sum() == 0:
    print("No recordings had enough windows on both sides of the wind-down boundary.")

## 2. Follow the first 20 usable minutes

Time-on-task analyses ask when EEG changes emerge and whether they persist. Saketh and colleagues tracked band power across breath-watching meditation. Here the clock starts at recording minute 2, and elapsed time is used because the onset of directed thought was unmarked. [Saketh et al., 2026](https://link.springer.com/article/10.1007/s12671-026-02790-1)

Compare four consecutive five-minute blocks that finish before wind-down. Each session must have at least 24 valid 10-second windows for **each band at all four sensors** in each block. Subtract that session's first-block median from its later medians. Each line shows one sensor's median change across the same sessions in every block. The final-block table shows how much the sessions varied. Compare when gamma, theta, alpha, and the other bands rise, fall, or level off.

In [ ]:
courses = []
for file, (_, fmt, end, windows, _) in recordings.items():
    stop = STARTUP_SECONDS + TIME_BLOCK_SECONDS * TIME_BLOCK_COUNT
    if stop > end - WIND_DOWN_SECONDS:
        continue
    interval = slice_windows(windows, STARTUP_SECONDS, stop)[EEG_COLUMNS]
    block_id = (interval.index - STARTUP_SECONDS) // TIME_BLOCK_SECONDS
    grouped = interval.groupby(block_id)
    medians = grouped.median().where(grouped.count() >= MIN_TIME_BLOCK_WINDOWS)
    medians = medians.reindex(range(TIME_BLOCK_COUNT))
    if medians.isna().any().any():
        continue
    changes = medians - medians.iloc[0]
    changes.index.name = "Block"
    courses.append(changes.reset_index().assign(File=file, Format=fmt))

if not courses:
    print("No sessions had enough coverage in all time blocks.")
else:
    time_courses = pd.concat(courses, ignore_index=True)
    formats = list(time_courses["Format"].unique())
    fig, axes = plt.subplots(len(BANDS), len(formats), figsize=(5.5 * len(formats), 9),
                             sharex=True, sharey="row", squeeze=False, layout="constrained")
    block_minutes = TIME_BLOCK_SECONDS / 60
    starts = STARTUP_SECONDS / 60 + np.arange(TIME_BLOCK_COUNT) * block_minutes
    labels = [f"{start:g}-{start + block_minutes:g}" for start in starts]
    for column, fmt in enumerate(formats):
        group = time_courses.loc[time_courses["Format"].eq(fmt)]
        median = group.groupby("Block")[EEG_COLUMNS].median()
        n_sessions = group["File"].nunique()
        axes[0, column].set_title(f"{FORMAT_LABELS[fmt]} | {n_sessions} sessions")
        for row, band in enumerate(BANDS):
            ax = axes[row, column]
            for sensor, color in SENSOR_COLORS.items():
                ax.plot(starts, median[f"{band}_{sensor}"], "o-", color=color, label=sensor)
            if row == 0:
                ax.legend(loc="best", ncols=2, fontsize=8)
            ax.axhline(0, color="0.45", linestyle="--", linewidth=1)
            if column == 0:
                ax.set_ylabel(band)
        axes[-1, column].set(xlabel="Minutes from recording start", xticks=starts, xticklabels=labels)
        print(f"{FORMAT_LABELS[fmt]} | {n_sessions} sessions | Last block minus first block")
        display(summarize_contrast(group.loc[group["Block"].eq(TIME_BLOCK_COUNT - 1)], EEG_COLUMNS))
    fig.suptitle("Band values relative to the first five-minute block")
    fig.supylabel("Change in recorded band value")
    plt.show()

## 3. Check gamma against head movement

Artifact checks matter for scalp beta and gamma. Recordings under neuromuscular blockade show that muscle activity can contribute to high-frequency scalp EEG. The Muse gyroscope measures head rotation, and facial tension can occur even with a quiet gyroscope. [Whitham et al., 2007](https://pubmed.ncbi.nlm.nih.gov/17574912/)

Within each complete five-minute practice block with enough gyroscope data, compare windows at or below the 20th percentile and at or above the 80th percentile of gyroscope magnitude. This compares windows from the same part of the session. Each signal needs at least four valid EEG windows per group. Use a median across eligible blocks for each session. Positive values mean the recorded band is higher during higher-motion windows. The table covers all bands at all four sensors. The plot shows gamma at each sensor and the right-minus-left difference for each pair. Files with constant or missing motion fields contribute no motion comparison.

In [ ]:
motion_rows = []
for file, (_, fmt, end, windows, _) in recordings.items():
    stop = end - WIND_DOWN_SECONDS
    complete_stop = STARTUP_SECONDS + (stop - STARTUP_SECONDS) // TIME_BLOCK_SECONDS * TIME_BLOCK_SECONDS
    practice = slice_windows(windows, STARTUP_SECONDS, complete_stop)
    blocks = practice.groupby((practice.index - STARTUP_SECONDS) // TIME_BLOCK_SECONDS)
    changes = []
    for _, block in blocks:
        paired = block.dropna(subset=["Gyro"])
        if len(paired) < MIN_MOTION_BLOCK_WINDOWS:
            continue
        low, high = paired["Gyro"].quantile([0.2, 0.8])
        if low >= high:
            continue
        quiet = paired.loc[paired["Gyro"].le(low), CONTRAST_COLUMNS]
        active = paired.loc[paired["Gyro"].ge(high), CONTRAST_COLUMNS]
        eligible = quiet.count().ge(4) & active.count().ge(4)
        if eligible[EEG_COLUMNS].any():
            changes.append((active.median() - quiet.median()).where(eligible))
    if changes:
        motion_rows.append({"File": file, "Format": fmt, "Blocks": len(changes),
                            **pd.DataFrame(changes).median().to_dict()})

motion_contrasts = pd.DataFrame(motion_rows, columns=["File", "Format", "Blocks", *CONTRAST_COLUMNS])
if motion_contrasts.empty:
    print("No recordings had enough varying gyroscope data for this comparison.")
else:
    for fmt, group in motion_contrasts.groupby("Format", sort=False):
        print(f"{FORMAT_LABELS[fmt]} | {len(group)} sessions, {group['Blocks'].sum()} paired blocks")
        display(summarize_contrast(group, CONTRAST_COLUMNS))
        gamma_columns = [f"Gamma_{sensor}" for sensor in SENSORS] + GAMMA_DIFFERENCES
        labels = [column.removeprefix("Gamma_").replace("_", " ") for column in gamma_columns]
        fig, ax = plt.subplots(figsize=(7.5, 4), layout="constrained")
        for y, column in enumerate(gamma_columns):
            values = group[column].dropna()
            if values.empty:
                continue
            lower, median, upper = values.quantile([0.25, 0.5, 0.75])
            ax.scatter(values, np.full(len(values), y), s=12, alpha=0.18, color="tab:blue")
            ax.hlines(y, lower, upper, color="tab:blue", linewidth=3)
            ax.plot(median, y, "o", color="tab:blue")
        ax.axvline(0, color="0.45", linestyle="--", linewidth=1)
        ax.set(yticks=range(len(labels)), yticklabels=labels,
               xlabel="Higher minus lower motion (recorded band units)",
               title=f"{FORMAT_LABELS[fmt]} | Gamma and head movement")
        ax.invert_yaxis()
        plt.show()

## 4. Explore alpha at all four sensors

Alpha often attenuates when people open their eyes, and some meditation studies report alpha changes with eye state held fixed. Your eyes-open wind-down gives us a useful transition to inspect. Its estimated start also marks the end of practice, so the two changes overlap. AF7 and AF8 sit at the front, and TP9 and TP10 sit behind the ears. This headset has no occipital electrodes for measuring the classic posterior alpha response directly. [Eyes-open versus eyes-closed EEG](https://pubmed.ncbi.nlm.nih.gov/17911042/) · [Nondirective meditation versus eyes-closed rest](https://pubmed.ncbi.nlm.nih.gov/19922249/)

### 4a. Does alpha change abruptly at wind-down?

Align each session to the estimated start of eyes-open wind-down. For each sensor, subtract the median alpha value in the preceding minute. Show the final minute of practice and the full two-minute wind-down. Lines show the median change across sessions. Shading spans the middle half of sessions at each time point. The table compares the first and last 30 seconds of wind-down with that same reference.

In [ ]:
ALPHA_COLUMNS = [f"Alpha_{sensor}" for sensor in SENSORS]
reference_seconds = 60
transition_frames = []
for file, (_, fmt, end, _, phase) in recordings.items():
    boundary = end - WIND_DOWN_SECONDS
    if boundary - reference_seconds < STARTUP_SECONDS:
        continue
    segment = slice_windows(phase, boundary - reference_seconds, end)[ALPHA_COLUMNS]
    reference = segment.loc[segment.index < boundary]
    sufficient_reference = reference.count() >= np.ceil(0.75 * reference_seconds / WINDOW_SECONDS)
    changes = segment - reference.median().where(sufficient_reference)
    changes.index = pd.Index(segment.index - boundary, name="Seconds")
    transition_frames.append(changes.reset_index().assign(File=file, Format=fmt))

if not transition_frames:
    print("No sessions contain both the alpha reference minute and wind-down.")
else:
    alpha_transition = pd.concat(transition_frames, ignore_index=True)
    for fmt, group in alpha_transition.groupby("Format", sort=False):
        by_time = group.groupby("Seconds")[ALPHA_COLUMNS]
        coverage = by_time.count() >= 5
        mid = by_time.median().where(coverage)
        lower = by_time.quantile(0.25).where(coverage)
        upper = by_time.quantile(0.75).where(coverage)
        fig, axes = plt.subplots(2, 2, figsize=(10, 5.5), sharex=True, sharey=True,
                                 layout="constrained")
        for ax, sensor in zip(axes.flat, SENSORS):
            key = f"Alpha_{sensor}"
            seconds = mid.index + WINDOW_SECONDS / 2
            color = SENSOR_COLORS[sensor]
            ax.fill_between(seconds, lower[key], upper[key], color=color, alpha=0.18)
            ax.plot(seconds, mid[key], color=color)
            ax.axvspan(0, WIND_DOWN_SECONDS, color="0.5", alpha=0.08)
            ax.axvline(0, color="0.4", linestyle="--", linewidth=1)
            ax.axhline(0, color="0.6", linewidth=1)
            ax.set_title(sensor)
        fig.suptitle(f"{FORMAT_LABELS[fmt]} | Alpha during eyes-open wind-down")
        fig.supxlabel("Seconds from estimated wind-down start")
        fig.supylabel("Alpha change from the previous minute")
        plt.show()

        summary = {}
        for label, start in [("First 30 s", 0), ("Last 30 s", WIND_DOWN_SECONDS - 30)]:
            section = group.loc[group["Seconds"].between(start, start + 30, inclusive="left")]
            by_file = section.groupby("File")[ALPHA_COLUMNS]
            per_file = by_file.median().where(by_file.count() >= np.ceil(0.75 * 30 / WINDOW_SECONDS))
            summary[f"{label} change"] = per_file.median()
            summary[f"{label} sessions"] = per_file.count()
        print(FORMAT_LABELS[fmt])
        display(pd.DataFrame(summary).round(3))

### 4b. Does high alpha occur in sustained stretches?

During practice, call a window high alpha when its value exceeds that sensor's 75th percentile for the session. Find consecutive runs of high windows and measure the share of high windows in runs lasting at least 30 seconds. Missing windows break a run. This asks how sustained the higher readings are within each sensor's own range. The percentile threshold is relative to each session and sensor.

In [ ]:
def sustained_alpha_fraction(signal):
    if signal.count() < 30:
        return np.nan
    high = signal.gt(signal.quantile(0.75))
    if not high.any():
        return np.nan
    adjacent = signal.index.to_series().diff().eq(WINDOW_SECONDS)
    run = (~(high & high.shift(fill_value=False) & adjacent)).cumsum()
    run_seconds = high.groupby(run).transform("sum") * WINDOW_SECONDS
    return 100 * run_seconds[high].ge(30).mean()


duration_rows = []
for file, (_, fmt, end, windows, _) in recordings.items():
    practice = slice_windows(windows, STARTUP_SECONDS, end - WIND_DOWN_SECONDS)
    sustained = practice[ALPHA_COLUMNS].apply(sustained_alpha_fraction)
    duration_rows.append({"File": file, "Format": fmt, **sustained.to_dict()})

alpha_duration = pd.DataFrame(duration_rows)
for fmt, group in alpha_duration.groupby("Format", sort=False):
    valid = group[ALPHA_COLUMNS]
    print(FORMAT_LABELS[fmt])
    display(pd.DataFrame({
        "Sessions": valid.count(), "Median % sustained": valid.median(),
        "25th percentile": valid.quantile(0.25), "75th percentile": valid.quantile(0.75),
    }).round(1))
    fig, ax = plt.subplots(figsize=(7.5, 3.3), layout="constrained")
    for y, sensor in enumerate(SENSORS):
        values = valid[f"Alpha_{sensor}"].dropna()
        if values.empty:
            continue
        color = SENSOR_COLORS[sensor]
        q1, midpoint, q3 = values.quantile([0.25, 0.5, 0.75])
        ax.scatter(values, np.full(len(values), y), s=12, alpha=0.16, color=color)
        ax.hlines(y, q1, q3, color=color, linewidth=3)
        ax.plot(midpoint, y, "o", color=color)
    ax.set(xlim=(-2, 102), yticks=range(len(SENSORS)), yticklabels=SENSORS,
           xlabel="High-alpha windows in runs of 30 seconds or longer (%)",
           title=f"{FORMAT_LABELS[fmt]} | Sustained high alpha")
    ax.invert_yaxis()
    plt.show()

### 4c. Which bands rise or fall with alpha?

Within each practice session and sensor, take the change between adjacent 10-second windows. Compare alpha changes with changes in delta, theta, beta, and gamma using Spearman correlation. It measures whether the ranks of the signed changes tend to move together. Require 30 paired changes and summarize session correlations by their median within each format. Differencing reduces the influence of slow trends and fixed offsets. Use delta here as a comparison signal. Blinks and eye movements can affect it, especially at frontal sensors. The coefficients describe co-movement of recorded band values. [Study of ocular artifacts by band and scalp site](https://www.sciencedirect.com/science/article/pii/S1388245700005411)

In [ ]:
comparison_bands = [band for band in BANDS if band != "Alpha"]
coupling_rows = []
for file, (_, fmt, end, windows, _) in recordings.items():
    practice = slice_windows(windows, STARTUP_SECONDS, end - WIND_DOWN_SECONDS)
    adjacent = practice.index.to_series().diff().eq(WINDOW_SECONDS)
    changes = practice[EEG_COLUMNS].diff().loc[adjacent]
    for sensor in SENSORS:
        alpha = f"Alpha_{sensor}"
        columns = [alpha, *[f"{band}_{sensor}" for band in comparison_bands]]
        correlation = changes[columns].corr(method="spearman", min_periods=30)
        coupling_rows.extend(
            {"File": file, "Format": fmt, "Sensor": sensor, "Band": band,
             "Correlation": correlation.loc[alpha, f"{band}_{sensor}"]}
            for band in comparison_bands
        )

alpha_coupling = pd.DataFrame(coupling_rows)
for fmt, group in alpha_coupling.groupby("Format", sort=False):
    summary = group.groupby(["Band", "Sensor"])["Correlation"].agg(["median", "count"])
    matrix = summary["median"].unstack().reindex(index=comparison_bands, columns=list(SENSORS))
    counts = summary["count"].unstack().reindex_like(matrix)
    matrix = matrix.where(counts >= 5)
    fig, ax = plt.subplots(figsize=(6.8, 3.6), layout="constrained")
    heatmap = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set(xticks=range(len(SENSORS)), xticklabels=SENSORS, yticks=range(len(comparison_bands)),
           yticklabels=comparison_bands, title=f"{FORMAT_LABELS[fmt]} | Changes with alpha")
    for (row, column), value in np.ndenumerate(matrix.to_numpy()):
        if np.isfinite(value):
            color = "white" if abs(value) > 0.65 else "black"
            ax.text(column, row, f"{value:+.2f}\nn={counts.iloc[row, column]}",
                    ha="center", va="center", color=color)
    fig.colorbar(heatmap, ax=ax, label="Median within-session Spearman correlation")
    plt.show()

## From session phases to labelled practice

For a future comparison of directed thought, quiet rest, and another style, record timestamped blocks with the same eye state and counterbalance their order over several days. An occasional report of focus, mind wandering, or drowsiness would let you align EEG windows to your experience. The current tables describe repeated sessions from your archive. The interquartile ranges show how much the sessions vary.

Phase synchrony, event-related potentials, and a fresh frequency spectrum require suitably sampled waveforms, reliable timing, and, for event-related analyses, event markers. The archived band summaries support the comparisons above without assuming that the CSV's RAW fields are a continuous waveform.